In [2]:
# Retrieve 5m OHLC data on BTC/USD.
# Endpoint does not require authentication,
# but has utility functions for authentication.

import http.client
import urllib.request
import urllib.parse
import hashlib
import hmac
import base64
import json
import time

def main():
   response = request(
      method="GET",
      path="/0/public/OHLC",
      query={
         "pair": "BTC/USD",
         "interval": 5,
      },
      environment="https://api.kraken.com",
   )
   print(response.read().decode())

def request(method: str = "GET", path: str = "", query: dict | None = None, body: dict | None = None, public_key: str = "", private_key: str = "", environment: str = "") -> http.client.HTTPResponse:
   url = environment + path
   query_str = ""
   if query is not None and len(query) > 0:
      query_str = urllib.parse.urlencode(query)
      url += "?" + query_str
   nonce = ""
   if len(public_key) > 0:
      if body is None:
         body = {}
      nonce = body.get("nonce")
      if nonce is None:
         nonce = get_nonce()
         body["nonce"] = nonce
   headers = {}
   body_str = ""
   if body is not None and len(body) > 0:
      body_str = json.dumps(body)
      headers["Content-Type"] = "application/json"
   if len(public_key) > 0:
      headers["API-Key"] = public_key
      headers["API-Sign"] = get_signature(private_key, query_str+body_str, nonce, path)
   req = urllib.request.Request(
      method=method,
      url=url,
      data=body_str.encode(),
      headers=headers,
   )
   return urllib.request.urlopen(req)

def get_nonce() -> str:
   return str(int(time.time() * 1000))

def get_signature(private_key: str, data: str, nonce: str, path: str) -> str:
   return sign(
      private_key=private_key,
      message=path.encode() + hashlib.sha256(
            (nonce + data)
         .encode()
      ).digest()
   )

def sign(private_key: str, message: bytes) -> str:
   return base64.b64encode(
      hmac.new(
         key=base64.b64decode(private_key),
         msg=message,
         digestmod=hashlib.sha512,
      ).digest()
   ).decode()


if __name__ == "__main__":
   main()

{"error":[],"result":{"BTC/USD":[[1778363700,"80782.1","80782.1","80782.0","80782.0","80782.0","0.10008820",74],[1778364000,"80782.1","80782.1","80765.0","80765.0","80780.6","0.27122251",77],[1778364300,"80765.1","80765.1","80729.9","80733.1","80745.8","0.35958458",100],[1778364600,"80733.1","80733.1","80725.9","80733.1","80732.9","1.03213097",79],[1778364900,"80733.0","80753.1","80733.0","80753.0","80738.7","0.30639732",86],[1778365200,"80753.0","80753.0","80750.6","80750.7","80752.6","0.33668772",61],[1778365500,"80750.7","80750.7","80739.7","80741.1","80748.1","8.13279276",123],[1778365800,"80740.9","80752.9","80662.3","80750.3","80729.3","16.99867821",377],[1778366100,"80750.2","80751.5","80733.6","80733.7","80743.4","0.12424054",69],[1778366400,"80733.6","80790.2","80733.6","80790.2","80753.4","0.71018795",100],[1778366700,"80790.2","80790.2","80761.3","80761.4","80785.2","0.26546439",88],[1778367000,"80761.4","80761.4","80750.7","80750.8","80759.7","0.16511066",86],[1778367300,"8

In [4]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

BASE_URL = "https://api.kraken.com"


def get_ohlc(pair="BTC/USD", interval=1440):
    params = urlencode({
        "pair": pair,
        "interval": interval,
    })

    url = f"{BASE_URL}/0/public/OHLC?{params}"

    try:
        with urlopen(url) as response:
            payload = json.load(response)

        if payload["error"]:
            raise RuntimeError(payload["error"])

        return payload["result"]

    except HTTPError as e:
        raise RuntimeError(f"HTTP error: {e.code}") from e

    except URLError as e:
        raise RuntimeError(f"Connection error: {e.reason}") from e


if __name__ == "__main__":
    ohlc = get_ohlc()

    print(json.dumps(ohlc, indent=2))

{
  "BTC/USD": [
    [
      1716336000,
      "70145.1",
      "70594.2",
      "68725.2",
      "69144.0",
      "69785.5",
      "2667.19208346",
      26690
    ],
    [
      1716422400,
      "69144.1",
      "70500.0",
      "66250.0",
      "67940.8",
      "68577.2",
      "3538.19855776",
      34662
    ],
    [
      1716508800,
      "67947.0",
      "69285.3",
      "66652.4",
      "68539.1",
      "68100.1",
      "1803.19203512",
      24678
    ],
    [
      1716595200,
      "68541.0",
      "69608.3",
      "68540.6",
      "69278.7",
      "69056.2",
      "627.90222520",
      14226
    ],
    [
      1716681600,
      "69278.7",
      "69461.2",
      "67971.1",
      "68476.4",
      "68926.9",
      "481.57199102",
      14241
    ],
    [
      1716768000,
      "68474.4",
      "70600.0",
      "68266.6",
      "69377.8",
      "69531.7",
      "1340.86348323",
      23310
    ],
    [
      1716854400,
      "69377.8",
      "69509.6",
      "67243.1",
    

In [11]:
from datetime import datetime, UTC
from urllib.parse import urlencode
from urllib.request import urlopen
from urllib.error import HTTPError, URLError
import json

BASE_URL = "https://api.kraken.com"


def to_datetime(timestamp: int) -> str:
    """Convert Unix timestamp to readable UTC datetime."""
    return datetime.fromtimestamp(timestamp, UTC).strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )


def get_ohlc(pair="BTC/USD", interval=1440, since=None):
    params = {
        "pair": pair,
        "interval": interval,
    }

    if since is not None:
        params["since"] = since

    url = f"{BASE_URL}/0/public/OHLC?{urlencode(params)}"

    try:
        with urlopen(url) as response:
            payload = json.load(response)

        if payload["error"]:
            raise RuntimeError(payload["error"])

        result = payload["result"]

        # Kraken returns the pair name dynamically
        pair_key = next(k for k in result.keys() if k != "last")

        candles = []

        for candle in result[pair_key]:
            candles.append({
                "timestamp": candle[0],
                "datetime": to_datetime(candle[0]),
                "open": float(candle[1]),
                "high": float(candle[2]),
                "low": float(candle[3]),
                "close": float(candle[4]),
                "vwap": float(candle[5]),
                "volume": float(candle[6]),
                "trades": int(candle[7]),
            })

        return candles

    except HTTPError as e:
        raise RuntimeError(f"HTTP error: {e.code}") from e

    except URLError as e:
        raise RuntimeError(f"Connection error: {e.reason}") from e


if __name__ == "__main__":
    candles = get_ohlc(
        pair="BTC/USD",
        interval=1440,
    )

    print(json.dumps(candles, indent=2))

[
  {
    "timestamp": 1716336000,
    "datetime": "2024-05-22 00:00:00 UTC",
    "open": 70145.1,
    "high": 70594.2,
    "low": 68725.2,
    "close": 69144.0,
    "vwap": 69785.5,
    "volume": 2667.19208346,
    "trades": 26690
  },
  {
    "timestamp": 1716422400,
    "datetime": "2024-05-23 00:00:00 UTC",
    "open": 69144.1,
    "high": 70500.0,
    "low": 66250.0,
    "close": 67940.8,
    "vwap": 68577.2,
    "volume": 3538.19855776,
    "trades": 34662
  },
  {
    "timestamp": 1716508800,
    "datetime": "2024-05-24 00:00:00 UTC",
    "open": 67947.0,
    "high": 69285.3,
    "low": 66652.4,
    "close": 68539.1,
    "vwap": 68100.1,
    "volume": 1803.19203512,
    "trades": 24678
  },
  {
    "timestamp": 1716595200,
    "datetime": "2024-05-25 00:00:00 UTC",
    "open": 68541.0,
    "high": 69608.3,
    "low": 68540.6,
    "close": 69278.7,
    "vwap": 69056.2,
    "volume": 627.9022252,
    "trades": 14226
  },
  {
    "timestamp": 1716681600,
    "datetime": "2024-05-26

In [3]:
from datetime import datetime, UTC
from urllib.parse import urlencode
from urllib.request import urlopen
from urllib.error import HTTPError, URLError
import json

BASE_URL = "https://api.kraken.com"


def to_datetime(timestamp: int) -> str:
    return datetime.fromtimestamp(timestamp, UTC).strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )


def get_ohlc(pair="BTC/USD", interval=1440, since=None):
    """
    Fetch OHLC candles from Kraken.

    Intervals:
        1      = 1 minute
        5      = 5 minutes
        15     = 15 minutes
        30     = 30 minutes
        60     = 1 hour
        240    = 4 hours
        1440   = 1 day
        10080  = 1 week
        21600  = 15 days
    """

    params = {
        "pair": pair,
        "interval": interval,
    }

    if since is not None:
        params["since"] = since

    url = f"{BASE_URL}/0/public/OHLC?{urlencode(params)}"

    try:
        with urlopen(url) as response:
            payload = json.load(response)

        if payload["error"]:
            raise RuntimeError(payload["error"])

        result = payload["result"]

        # Pair key is dynamic
        pair_key = next(k for k in result.keys() if k != "last")

        candles = []

        for candle in result[pair_key]:
            candles.append({
                "timestamp": candle[0],
                "datetime": to_datetime(candle[0]),
                "open": float(candle[1]),
                "high": float(candle[2]),
                "low": float(candle[3]),
                "close": float(candle[4]),
                "vwap": float(candle[5]),
                "volume": float(candle[6]),
                "trades": int(candle[7]),
            })

        return candles

    except HTTPError as e:
        raise RuntimeError(f"HTTP error: {e.code}") from e

    except URLError as e:
        raise RuntimeError(f"Connection error: {e.reason}") from e


if __name__ == "__main__":
    daily_btc = get_ohlc(
        pair="BTC/USD",
        interval=1440,   # daily candles
    )

    data = json.dumps(daily_btc, indent=2)

In [5]:
import pandas as pd

# create dataframe
df = pd.DataFrame(json.loads(data))

# convert datetime column to pandas datetime
df["datetime"] = pd.to_datetime(df["datetime"])

# optional: use datetime as index
df.set_index("datetime", inplace=True)

print(df.head())

                            timestamp     open     high      low    close  \
datetime                                                                    
2024-05-27 00:00:00+00:00  1716768000  68474.4  70600.0  68266.6  69377.8   
2024-05-28 00:00:00+00:00  1716854400  69377.8  69509.6  67243.1  68324.9   
2024-05-29 00:00:00+00:00  1716940800  68325.0  68843.3  67100.0  67575.4   
2024-05-30 00:00:00+00:00  1717027200  67575.4  69550.0  67099.9  68365.0   
2024-05-31 00:00:00+00:00  1717113600  68365.0  69000.0  66652.4  67498.1   

                              vwap       volume  trades  
datetime                                                 
2024-05-27 00:00:00+00:00  69531.7  1340.863483   23310  
2024-05-28 00:00:00+00:00  68161.5  1727.620024   25966  
2024-05-29 00:00:00+00:00  67781.8  1916.841463   21965  
2024-05-30 00:00:00+00:00  68503.7  2098.190112   25159  
2024-05-31 00:00:00+00:00  67756.6  1214.317517   22910  


In [2]:
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

BASE_URL = "https://api.freecryptoapi.com/v1"
API_KEY = "tk3yaqbm1y5gbjjzhp8f"


def get_bitcoin_news(limit=10):
    """
    Fetch latest Bitcoin news.
    """

    params = urlencode({
        "keyword": "bitcoin",
        "limit": limit,
    })

    url = f"{BASE_URL}/getNews?{params}"

    headers = {
        "Authorization": f"Bearer {API_KEY}"
    }

    request = Request(url, headers=headers)

    try:
        with urlopen(request) as response:
            return json.load(response)

    except HTTPError as e:
        raise RuntimeError(f"HTTP error: {e.code}") from e

    except URLError as e:
        raise RuntimeError(f"Connection error: {e.reason}") from e


if __name__ == "__main__":
    news = get_bitcoin_news(limit=5)

    print(json.dumps(news, indent=2))

RuntimeError: HTTP error: 403

In [15]:
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

API_KEY = "tk3yaqbm1y5gbjjzhp8f"

BASE_URL = "https://api.freecryptoapi.com"


def get_bitcoin_news(limit=5):
    params = urlencode({
        "keyword": "bitcoin",
        "limit": limit,
    })

    url = f"{BASE_URL}/v1/getNews?{params}"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json",
        "User-Agent": "Python"
    }

    request = Request(
        url=url,
        headers=headers,
        method="GET"
    )

    try:
        with urlopen(request) as response:
            data = json.load(response)

        return data

    except HTTPError as e:
        print("STATUS:", e.code)
        print(e.read().decode())
        raise

    except URLError as e:
        print("Connection error:", e.reason)
        raise


if __name__ == "__main__":
    news = get_bitcoin_news()

    print(json.dumps(news, indent=2))

{
  "status": false,
  "error": "No access. Please upgrade your subscription",
  "exec_time": 0.2689
}


In [16]:
import json
from urllib.request import urlopen

url = (
    "https://min-api.cryptocompare.com/data/v2/news/"
    "?lang=EN&categories=BTC"
)

with urlopen(url) as response:
    data = json.load(response)

for article in data["Data"][:5]:
    print(article["title"])

KeyError: slice(None, 5, None)

In [19]:
import feedparser
from datetime import datetime

RSS_FEEDS = [
    "https://cointelegraph.com/rss/tag/bitcoin",
    "https://decrypt.co/feed",
    "https://bitcoinmagazine.com/.rss/full/",
]


def fetch_bitcoin_news(limit=100):
    articles = []

    for feed_url in RSS_FEEDS:
        feed = feedparser.parse(feed_url)

        for entry in feed.entries:
            title = entry.get("title", "")

            # Basic BTC relevance filter
            if "bitcoin" not in title.lower() and "btc" not in title.lower():
                continue

            articles.append({
                "title": title,
                "link": entry.get("link"),
                "published": entry.get("published"),
                "source": feed.feed.get("title"),
            })

    # Sort newest first
    articles.sort(
        key=lambda x: x["published"],
        reverse=True,
    )

    return articles[:limit]


if __name__ == "__main__":
    news = fetch_bitcoin_news()

    for article in news:
        print("\n---")
        print(article["title"])
        print(article["source"])
        print(article["published"])
        print(article["link"])


---
Why eBay Should Ignore GameStop and Use Bitcoin to Save $1.2 Billion in Transaction Costs
Bitcoin Magazine
Wed, 06 May 2026 22:58:57 +0000
https://bitcoinmagazine.com/bitcoin-for-corporations/ebay-ignore-gamestop-bitcoin

---
Boltz Launches Non-Custodial USDC Swaps, Bridging Bitcoin Directly to Circle’s Regulated Dollar
Bitcoin Magazine
Wed, 06 May 2026 17:00:00 +0000
https://bitcoinmagazine.com/business/boltz-launches-non-custodial-usdc-swaps-bridging-bitcoin-directly-to-circles-regulated-dollar

---
Strategy Opens Door to Bold Bitcoin Sales Pivot Unlocking $2.2 Billion Tax Benefit
Bitcoin Magazine
Wed, 06 May 2026 11:46:07 +0000
https://bitcoinmagazine.com/bitcoin-for-corporations/sstrategy-mstr-bitcoin-sales-pivot

---
Bitcoin funding rates turn positive: Is BTC rally to $85K next?
Cointelegraph.com News
Tue, 12 May 2026 02:42:27 +0000
https://cointelegraph.com/markets/bitcoin-funding-rates-turn-positive-is-a-btc-rally-to-85k-next?utm_source=rss_feed&utm_medium=rss_tag_bitcoin&

In [1]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode


BASE_URL = "https://cryptocurrency.cv/api"


def get_historical_bitcoin_news(limit=10):
    params = urlencode({
        "q": "bitcoin",
        "limit": limit,
    })

    url = f"{BASE_URL}/search?{params}"

    with urlopen(url) as response:
        return json.load(response)


if __name__ == "__main__":
    news = get_historical_bitcoin_news()

    for article in news["results"][:5]:
        print("\n---")
        print(article["title"])
        print(article.get("published_at"))
        print(article.get("url"))

HTTPError: HTTP Error 402: Payment Required

In [2]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode


BASE_URL = "https://www.sharpe.ai/api/news/feed"


def fetch_bitcoin_news(limit=10):
    params = urlencode({
        "coin": "bitcoin",
        "limit": limit,
    })

    url = f"{BASE_URL}?{params}"

    with urlopen(url) as response:
        payload = json.load(response)

    articles = []

    for article in payload:
        articles.append({
            "title": article.get("title"),
            "url": article.get("url"),
            "source": article.get("source"),
            "published": article.get("published_at"),
            "summary": article.get("summary"),
        })

    return articles


if __name__ == "__main__":
    articles = fetch_bitcoin_news()

    for article in articles:
        print("\n---")
        print(article["title"])
        print(article["published"])
        print(article["url"])

URLError: <urlopen error [Errno -2] Name or service not known>

In [3]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode


BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"


def fetch_bitcoin_news(max_records=10):
    params = urlencode({
        "query": "bitcoin",
        "mode": "artlist",
        "format": "json",
        "maxrecords": max_records,
    })

    url = f"{BASE_URL}?{params}"

    with urlopen(url) as response:
        data = json.load(response)

    return data


if __name__ == "__main__":
    news = fetch_bitcoin_news()

    for article in news["articles"]:
        print("\n---")
        print(article["title"])
        print(article["url"])
        print(article["seendate"])

URLError: <urlopen error [Errno -5] No address associated with hostname>

In [4]:
params = {
    "query": "bitcoin",
    "mode": "artlist",
    "format": "json",
    "startdatetime": "20240101000000",
    "enddatetime": "20240201000000",
}

In [7]:
import json
import time
from urllib.request import Request, urlopen
from urllib.parse import urlencode
from urllib.error import HTTPError

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"


def fetch_bitcoin_news(max_records=10, retries=3):
    params = urlencode({
        "query": "bitcoin",
        "mode": "artlist",
        "format": "json",
        "maxrecords": max_records,
    })

    url = f"{BASE_URL}?{params}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    request = Request(url, headers=headers)

    for attempt in range(retries):
        try:
            with urlopen(request, timeout=20) as response:
                data = json.loads(
                    response.read().decode()
                )

            return data

        except HTTPError as e:
            if e.code == 429:
                wait_time = 2 ** attempt

                print(
                    f"Rate limited. "
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

            else:
                raise

    raise RuntimeError(
        "Failed after multiple retries."
    )


if __name__ == "__main__":
    news = fetch_bitcoin_news()

    for article in news["articles"]:
        print("\n---")
        print(article["title"])
        print(article["url"])
        print(article["seendate"])


---
Why Bitcoin Still Looks Like Crypto Best Generational Wealth Bet
https://www.fool.com/investing/2026/05/02/why-bitcoin-still-looks-like-cryptos-best-generati/?source=iedfolrf0000001
20260502T141500Z

---
  비트코인 , 2030년까지 10배 오른다 … 돈나무 언니  의 대폭등 전망 , 왜 ? 
https://biz.heraldcorp.com/article/10730133
20260502T141500Z

---
How Eric Trump American Bitcoin Lost $500 Million In Shareholder Value In 8 Months - American Bitcoin ( 
https://www.benzinga.com/crypto/cryptocurrency/26/04/52108321/how-eric-trumps-american-bitcoin-lost-500-million-in-shareholder-value-in-8-months
20260429T133000Z

---
Tim Draper Says Firms  Irresponsible  To Not Hold Bitcoin as Bullish Predictions Surge At Bitcoin 2026
https://finance.yahoo.com/markets/crypto/articles/tim-draper-says-firms-irresponsible-141715821.html
20260429T223000Z

---
This  Set It and Forget It  Cryptocurrency Could Make You a Multimillionaire With Almost No Effort
https://finance.yahoo.com/markets/crypto/articles/set-forget-cryptocurrency-c

In [10]:
import json
from urllib.request import Request, urlopen
from urllib.parse import urlencode

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"


def fetch_bitcoin_news_by_date(date):
    # date format: YYYYMMDD
    start = date + "000000"
    end = date + "235959"

    params = urlencode({
        "query": "bitcoin",
        "mode": "artlist",
        "format": "json",
        "startdatetime": start,
        "enddatetime": end,
        "maxrecords": 100
    })

    url = f"{BASE_URL}?{params}"

    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())


if __name__ == "__main__":
    data = fetch_bitcoin_news_by_date("20240101")

    for a in data.get("articles", []):
        print(a)
        print(a["seendate"], a["title"])

HTTPError: HTTP Error 429: Too Many Requests

In [19]:
import json
from urllib.request import urlopen, Request
from urllib.parse import urlencode

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"


def fetch_bitcoin_news_english(date=None):
    params = {
        "query": "bitcoin",
        "mode": "artlist",
        "format": "json",
        "maxrecords": 100
    }

    # optional date filter
    if date:
        params["startdatetime"] = date + "000000"
        params["enddatetime"] = date + "235959"

    url = f"{BASE_URL}?{urlencode(params)}"

    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())


if __name__ == "__main__":
    data = fetch_bitcoin_news_english("20260516")

    for a in data.get("articles", []):
        if a["language"] == "English":
            #print(a)
            print(a["seendate"], a["title"])
            

HTTPError: HTTP Error 429: Too Many Requests

In [22]:
import json
import time

from urllib.request import urlopen, Request
from urllib.parse import urlencode
from urllib.error import HTTPError


BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"


def fetch_bitcoin_news_english(date=None):

    query = "bitcoin"

    params = {
        "query": query,
        "mode": "ArtList",
        "format": "json",
        "maxrecords": 10,
    }

    if date:
        params["startdatetime"] = f"{date}000000"
        params["enddatetime"] = f"{date}235959"

    url = f"{BASE_URL}?{urlencode(params)}"

    print(url)

    req = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    try:

        with urlopen(req, timeout=20) as r:
            return json.loads(r.read().decode())

    except HTTPError as e:

        print("STATUS:", e.code)

        body = e.read().decode()

        print(body)

        if e.code == 429:

            retry_after = e.headers.get("Retry-After")

            print("Retry-After:", retry_after)

        raise


if __name__ == "__main__":

    time.sleep(2)

    data = fetch_bitcoin_news_english("20260516")

    for a in data.get("articles", []):

        print(a["seendate"], a["title"])

https://api.gdeltproject.org/api/v2/doc/doc?query=bitcoin&mode=ArtList&format=json&maxrecords=10&startdatetime=20260516000000&enddatetime=20260516235959
20260516T120000Z Strategy Quietly Confirms Shock Plan To Sell Bitcoin , Sparking Sudden Price Crash  Panic  
20260516T153000Z Prediction : Bitcoin Will Hit $1 Million -- Here the Timeline
20260516T010000Z 이더리움 늘리고 , 코인株 적극 투자 … 월가 1분기 자산보유공시 보니
20260517T000000Z AI Claude  cứu  thành công chiếc ví Bitcoin quên mật khẩu sau hơn một thập kỷ
20260516T134500Z Kriptoda iki günlük şok dalgası ! Bitcoin 78 bin doların altına çakıldı ... 
20260516T224500Z Is the 2 - Year Treasury at 4 . 09 % Why Bitcoin ( BTC ) Cant Break Out ? 
20260516T223000Z Why Bitcoin Price Could Be Forming A Consolidation Structure Around $80 , 000
20260516T223000Z Analyst Says Dont Buy Bitcoin Until This Happens
20260516T210000Z Is the 2 - Year Treasury at 4 . 09 % Why Bitcoin ( BTC ) Cant Break Out ? 
20260516T134500Z Bitcoin 78 bin doların altına geriledi - Son Dakika

In [ ]:
from ollama import chat

response = chat(
    model='phi4-mini',
    messages=[{'role': 'system', 'content': 'always speak french'},
        {'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

In [26]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="phi4-mini",
    temperature=0.7,
)

response = llm.invoke("Write a long haiku about programming.")

print(response.content)

In code's serene flow,
Logic weaves through lines unseen—
Silence births new worlds.

Debugging dawns light,
Binary dreams awaken.
Code dances, time fades—transcendent peace found in creation’s pulse.


In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} like I'm 10 years old."
)

model = ChatOllama(model="phi4-mini")

chain = prompt | model

result = chain.invoke({"topic": "black holes"})

print(result.content)

Alright, imagine you have a super huge stretchy rubber band that can stretch infinitely far without ever snapping back into place— that's kinda what space is! But sometimes objects get so heavy they squish this infinite stretching until it's not just stretched out anymore; it becomes invisible and stops letting anything else escape from or even touch. That's sort of like how black holes work in the universe: they're places where gravity pulls everything straight towards them, including light itself—which makes stars appear to disappear! It's a mystery because once you get past this rubber band-like space (the "event horizon"), there's no way back and nothing we can see anymore—hence they are called 'black' holes.
